# VeloceReduction — one observing night

**Data flow**

`DetectorFrame → OrderGeometry → OrderMatrix → ExtractionResult`

For `extraction_mode="fibre"`, `FibreGeometry` is an additional Flat-derived input to the final extraction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

from velocereduction import __version__, ReductionConfig
from velocereduction import observations, detector, orders, fibres, flat, extraction, wavelength
from velocereduction.config import prepare_reduction, setup_logging

night = "001122"
# night = "260703"
config = ReductionConfig(
    night=night,
    extraction_mode="fibre",   # "summed" for summing information along cross-dispersion direction; "fibre" for extracting each science fibre separately
    diagnostics="full",
    log_level="DEBUG",
    overwrite=False,
)
paths = prepare_reduction(config, __version__)
logger = setup_logging(config, paths)
print(paths.root)


## 1. Identify observations

The observing log and FITS headers are reconciled first. This stage only decides what data are available and which CCDs should be used.

In [ ]:
reduction_input = observations.identify_observations(config, paths)
display(reduction_input)
print(f"{len(reduction_input)} observations selected for {night}")

## 2. Detector registration

Detector shifts are measured in `detector.py` relative to the reference night. The compact result is written as `detector_shifts_YYMMDD.fits`.


In [ ]:
detector_shifts = detector.measure_detector_shifts(reduction_input, config, paths)
display(detector_shifts)
fits.info(paths.detector_shifts)

## 3. Combine Flats in memory

The individual Flat `DetectorFrame`s are normalized and combined. The full 4112×4096 combined Flat is intentionally **not** saved; it is only an intermediate used to determine geometry and compact 1-D response products.


In [ ]:
combined_flats = flat.combine_flat_frames(reduction_input, config)
for ccd, frame in combined_flats.items():
    print(f"CCD{ccd}: image={frame.image.shape}, finite={np.mean(np.isfinite(frame.image)):.3%}")

## 4. Determine `OrderGeometry`

The reference-night geometry supplies the starting locations. The current Flat and measured detector shifts refine the trace and named cross-dispersion regions. The persistent product has one table row per physical echelle order.


In [ ]:
order_geometry = orders.determine_order_geometry(
    reduction_input, combined_flats, detector_shifts, config, paths
)
order_table = orders.order_geometry_table(order_geometry)
display(order_table[:10])
fits.info(paths.order_geometry)
print(paths.order_geometry)

## 5. Optional compact `FibreGeometry`

For fibre extraction only, the Flat order matrices are fitted at sparse dispersion locations. The saved model contains polynomial coefficients for bundle offset, fibre separation and common Gaussian width, plus one constant offset for each science/sky fibre. The evaluated 4112-row centres are never written to disk.


In [ ]:
flat_order_matrices = flat.extract_flat_order_matrices(combined_flats, order_geometry)
fibre_geometry = {}

if config.extraction_mode == "fibre":
    reference_file = paths.reference_product("fibre_geometry", config.reference_night)
    reference_geometry = fibres.load_fibre_geometry(reference_file) if reference_file.exists() else {}

    if paths.fibre_geometry.exists() and not config.overwrite:
        fibre_geometry = fibres.load_fibre_geometry(paths.fibre_geometry)
    else:
        fibre_geometry = fibres.fit_fibre_geometries(
            flat_order_matrices, config, reference_geometries=reference_geometry
        )
        fibres.save_fibre_geometry(paths.fibre_geometry, fibre_geometry, config)

    summary = fibres.summarise_fibre_geometry(fibre_geometry)
    display(summary)
    fits.info(paths.fibre_geometry)
    with fits.open(paths.fibre_geometry) as hdul:
        display(Table(hdul["ORDER_MODEL"].data)[:8])
        display(Table(hdul["FIBRE_OFFSETS"].data)[:30])

    fibres.save_fibre_diagnostics(fibre_geometry, flat_order_matrices, config, paths)


## 6. Summed and fibre Flat calibrations

For the summed path, the science aperture gives one 4112-pixel Flat spectrum per order; a broad Gaussian-smoothed version is the large-scale illumination/blaze-like shape and their ratio is the small-scale response.

In fibre mode, each extracted fibre is treated independently. `response_fibres` also carries relative fibre-throughput information with respect to the median science-fibre smooth Flat. No 4112×81 multiplicative response map is applied to science/calibration order matrices.


In [ ]:
flat_calibrations = flat.build_flat_calibrations(
    flat_order_matrices, fibre_geometry, config, paths
)

for filename in (
    paths.flat_summed,
    paths.flat_smooth_summed,
    paths.response_summed,
):
    print(filename.name)
    fits.info(filename)

if config.extraction_mode == "fibre":
    for filename in (
        paths.flat_fibres,
        paths.flat_smooth_fibres,
        paths.response_fibres,
    ):
        print(filename.name)
        fits.info(filename)


### Checkpoint: direct summed Flat versus recombined fibres

This is an important independent QA test. The direct summed extraction is retained as the minimally model-dependent reference; wavelength-dependent structure in the fibre/summed ratio can reveal imperfect fibre geometry or deblending.


In [ ]:
if config.extraction_mode == "fibre":
    name = next(name for name in flat_calibrations if name.startswith("ccd_2_"))
    product = flat_calibrations[name]
    geometry = fibre_geometry[name]
    science_idx = geometry.component_indices if False else [geometry.components.index(f) for f in geometry.components if isinstance(f, (int, np.integer))]
    recombined = np.nansum(product.fibre_flat[:, science_idx], axis=1)
    scale = np.nanmedian(product.summed_flat / recombined)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(product.summed_flat, label="direct summed Flat")
    ax.plot(recombined * scale, label="recombined science fibres")
    ax.set(title=name, xlabel="Dispersion pixel", ylabel="Flat counts")
    ax.set_ylim(-1,3)
    ax.legend()
    plt.show()


## 7. Extract calibration spectra in detector coordinates

The same fixed `OrderGeometry` is used for SimTh, SimLC and summed FibTh extraction. Fibre mode additionally extracts the 19 science-fibre FibTh spectra using the fixed `FibreGeometry`. These spectra are deliberately left in detector-pixel coordinates for the next wavelength-calibration stage.


In [ ]:
calibration_exposures = extraction.extract_calibration_exposures(
    reduction_input, order_geometry, fibre_geometry, config
)
calibration_files = extraction.save_calibration_exposures(
    calibration_exposures, paths.calibrations, config.night, overwrite=True
)
for filename in calibration_files:
    print(filename.relative_to(paths.root))


# STOP HERE,
## BECAUSE WE ONLY DO THE WAVELENGTH CALIBRATION ONCE ORDERS/FIBRES WORK WELL!

In [ ]:
import sys
sys.exit()

## 8. Extract calibrations and fit wavelength model


In [ ]:
model_file = paths.wavelength_mode / "wavelength_model.fits"
if model_file.exists() and not config.overwrite:
    calibration_exposures = None
    wavelength_model = wavelength.load_model(model_file)
else:
    calibration_exposures = extraction.extract_calibration_exposures(
        reduction_input, nightly_tramlines, flat_products, config
    )
    extraction.save_calibration_exposures(
        calibration_exposures, paths.wavelength_mode / "extracted", overwrite=True
    )
    wavelength_model = wavelength.build_night_model(
        calibration_exposures, detector_shifts, config, paths
    )

display(wavelength.model_summary(wavelength_model))


The wavelength hierarchy is:

- summed FibTh: absolute science-bundle anchor;
- SimLC on CCD2/3: independent high-precision temporal drift;
- per-fibre FibTh: low-order residual correction for each of the 19 science fibres.


## 9. Science extraction


In [ ]:
science_exposures = science.extract_science_exposures(
    reduction_input, nightly_tramlines, flat_products,
    wavelength_model, config, paths
)
print(f"{len(science_exposures)} science CCD exposures")

if science_exposures:
    exposure = science_exposures[0]
    order = exposure.orders[len(exposure.orders) // 2]
    plt.figure(figsize=(10, 3))
    plt.plot(order.barycentric_wavelength_nm, order.flux)
    plt.xlabel("Barycentric wavelength / nm")
    plt.ylabel("Flux")
    plt.title(f"{exposure.object_name} — CCD{exposure.ccd}, order {order.order}")
    plt.show()


## Automatic QA products

With `diagnostics='basic'`, each stage above has already written the compact nightly QA figures to `figures/`. `full` additionally writes per-order diagnostics to `debug/`.


In [ ]:
if config.diagnostics != 'none':
    for filename in sorted(paths.figures.rglob('*.png')):
        print(filename.relative_to(paths.root))


## One-call equivalent


In [ ]:
# from velocereduction import pipeline
# state = pipeline.reduce_night(config, version=__version__)
